In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 1. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / 'train_split.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / 'val_split.parquet')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [2]:
# ============================================================
# 2. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
# print(len(wl_cols), len(rain_cols), len(env_cols), len(num_cols))

In [3]:
# ============================================================
# 3. 종속변수 설정
# ============================================================

y_train = train[target_col]
y_val = val[target_col]

In [4]:
# ============================================================
# 4. station_id 원핫인코딩
# ============================================================
# OHE 포함한게 조금이나마 나은 결과, 포함해서 모델링 시행

from sklearn.preprocessing import OneHotEncoder

# 원핫인코더 생성
ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train[['station_id']])
val_st_id_ohe = ohe.transform(val[['station_id']])

In [5]:
# ============================================================
# 5. 모델3: 수위, 강우, 유역환경 변수군 ablation 모델 만들기
# ============================================================

# 5-1. 수위, 강우, 유역환경 변수군 ablation 데이터셋 컬럼 정의

sets_ablation_cols = {
    'no_wl_cols' : [col for col in num_cols if col not in wl_cols],
    'no_rain_cols' : [col for col in num_cols if col not in rain_cols],
    'no_env_cols' : [col for col in num_cols if col not in env_cols]
}

print(sets_ablation_cols.items())  # datasets의 이름과 내용물이 같이 나옴(반환값 두개)

dict_items([('no_wl_cols', ['imerg_rain_1h', 'imerg_rain_6h', 'imerg_rain_24h', 'sfc_rain_1h', 'sfc_rain_6h', 'sfc_rain_24h', 'radar_reflectivity_mean', 'radar_reflectivity_p95', 'radar_reflectivity_ge20_fraction', 'cat_area', 'wsarea', 'mean_elevation', 'mean_slope', 'urban_ratio', 'forest_ratio', 'stream_order']), ('no_rain_cols', ['wl_t', 'wl_t_minus_1h', 'wl_t_minus_3h', 'wl_t_minus_6h', 'wl_t_minus_12h', 'wl_t_minus_24h', 'cat_area', 'wsarea', 'mean_elevation', 'mean_slope', 'urban_ratio', 'forest_ratio', 'stream_order']), ('no_env_cols', ['wl_t', 'wl_t_minus_1h', 'wl_t_minus_3h', 'wl_t_minus_6h', 'wl_t_minus_12h', 'wl_t_minus_24h', 'imerg_rain_1h', 'imerg_rain_6h', 'imerg_rain_24h', 'sfc_rain_1h', 'sfc_rain_6h', 'sfc_rain_24h', 'radar_reflectivity_mean', 'radar_reflectivity_p95', 'radar_reflectivity_ge20_fraction'])])


In [6]:
# 5-2 모델링: 변수군 3종 ablation
# ohe와 합치기~모델링~평가 및 검증까지 for 문안에 한번에 만들기

# 3-1. 변수군별 제외 모델링

from scipy.sparse import hstack
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error

# 해당 데이터셋 ohe와 합치기
result = {}  # 조건별 rmse 저장용 딕셔너리

for name, cols in sets_ablation_cols.items(): # name: 변수군의 이름, cols: 컬럼군
    print(name)
    X_train = hstack([
        train[cols],
        train_st_id_ohe
    ]).tocsr()
    
    X_val = hstack([
        val[cols],
        val_st_id_ohe
    ]).tocsr()

    # 모델링
    model = LGBMRegressor(
        random_state = 42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    # 성능평가
    rmse = root_mean_squared_error(y_val, pred)

    # 결과에 더하기
    result[name] = round(rmse, 4)  # result는 딕셔너리, []는 key가 됨
    print()

print(result)

no_wl_cols
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024332 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4930
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 481
[LightGBM] [Info] Start training from score 0.111701

no_rain_cols
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.101790 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4202
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 478
[LightGBM] [Info] Start training from score 0.111701

no_env_cols
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.023118 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4718
[LightGBM] [Info] Number of data points in the train set: 616331, number of us

In [7]:
# ============================================================
# 6. 모델3: 수위 개별변수별 ablation 모델링
# ============================================================

wl_ablation_col = {
    'no_wl_t' : 'wl_t',
    'no_wl_t_minus_1h' : 'wl_t_minus_1h',
    'no_wl_t_minus_3h' : 'wl_t_minus_3h',
    'no_wl_t_minus_6h' : 'wl_t_minus_6h',
    'no_wl_t_minus_12h' : 'wl_t_minus_12h',
    'no_wl_t_minus_24h' : 'wl_t_minus_24h',
}


for name, col in wl_ablation_col.items(): # name: 변수군의 이름, cols: 컬럼군
    print(name)
    X_train = hstack([
        train[[c for c in num_cols if c != col]],
        train_st_id_ohe
    ]).tocsr()
    
    X_val = hstack([
        val[[c for c in num_cols if c != col]],
        val_st_id_ohe
    ]).tocsr()

    # 모델링
    model = LGBMRegressor(
        random_state = 42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    # 성능평가
    rmse = root_mean_squared_error(y_val, pred)

    # 결과에 더하기
    result[name] = round(rmse, 4)  # result는 딕셔너리, []는 key가 됨

print(result)

no_wl_t
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009469 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6205
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 486
[LightGBM] [Info] Start training from score 0.111701
no_wl_t_minus_1h
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013062 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6205
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 486
[LightGBM] [Info] Start training from score 0.111701
no_wl_t_minus_3h
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.039514 seconds.
You can set `force_col_wise=true` to r

In [8]:
# 6(추가). 수위변수군 전체 영향력 >> 개별변수 영향력의 합  -> 수위변수군 변수간 상호작용 확인: 상관계수 필요
print("피어슨 상관계수: 수위변수군")
print(
    train[wl_cols]
    .corr(
        method='pearson'
    )
    .round(2)
    .to_csv(sep='\t')
)
print()

print("스피어만 상관계수: 수위변수군")
print(
    train[wl_cols]
    .corr(
        method='spearman'
    )
    .round(2)
    .to_csv(sep='\t')
)
# 정리 필요
# 수위변수군 내부적으로 강력한 상관성이 확인됨
# 그러나 개별 변수의 종속변수에 대한 상관성은 미미함
# -> 개별값보다 시차별 수위 변화에 대한 정보가 예측에 활용됐을 가능성
# -> 차분에 의한 파생변수 생성 검토 필요


피어슨 상관계수: 수위변수군
	wl_t	wl_t_minus_1h	wl_t_minus_3h	wl_t_minus_6h	wl_t_minus_12h	wl_t_minus_24h
wl_t	1.0	1.0	1.0	1.0	1.0	1.0
wl_t_minus_1h	1.0	1.0	1.0	1.0	1.0	1.0
wl_t_minus_3h	1.0	1.0	1.0	1.0	1.0	1.0
wl_t_minus_6h	1.0	1.0	1.0	1.0	1.0	1.0
wl_t_minus_12h	1.0	1.0	1.0	1.0	1.0	1.0
wl_t_minus_24h	1.0	1.0	1.0	1.0	1.0	1.0


스피어만 상관계수: 수위변수군
	wl_t	wl_t_minus_1h	wl_t_minus_3h	wl_t_minus_6h	wl_t_minus_12h	wl_t_minus_24h
wl_t	1.0	1.0	0.98	0.96	0.96	0.94
wl_t_minus_1h	1.0	1.0	0.99	0.96	0.96	0.94
wl_t_minus_3h	0.98	0.99	1.0	0.98	0.96	0.94
wl_t_minus_6h	0.96	0.96	0.98	1.0	0.96	0.94
wl_t_minus_12h	0.96	0.96	0.96	0.96	1.0	0.98
wl_t_minus_24h	0.94	0.94	0.94	0.94	0.98	1.0



In [12]:
# ============================================================
# 7. 모델3: 강우 세부변수군(imerg, sfc, radar) ablation 모델링
# ============================================================
# 3-3. 강우변수 세분화해서 영향력 확인
rain_ablation_cols = {
    'no_imerg' : [
        'imerg_rain_1h',
        'imerg_rain_6h',
        'imerg_rain_24h'
    ],
    'no_sfc' : [
        'sfc_rain_1h',
        'sfc_rain_6h',
        'sfc_rain_24h'
    ],
    'no_radar_reflect' : [
        'radar_reflectivity_mean',
        'radar_reflectivity_p95',
        'radar_reflectivity_ge20_fraction'
    ]
}

for name, cols in rain_ablation_cols.items(): # name: 변수군의 이름, cols: 컬럼군
    print(name)
    X_train = hstack([
        train[[c for c in num_cols if c not in cols]],
        train_st_id_ohe
    ]).tocsr()
    
    X_val = hstack([
        val[[c for c in num_cols if c not in cols]],
        val_st_id_ohe
    ]).tocsr()

    # 모델링
    model = LGBMRegressor(
        random_state = 42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    # 성능평가
    rmse = root_mean_squared_error(y_val, pred)

    # 결과에 더하기
    result[name] = round(rmse, 4)  # result는 딕셔너리, []는 key가 됨

no_imerg
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.043347 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5695
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 484
[LightGBM] [Info] Start training from score 0.111701
no_sfc
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009943 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5727
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 484
[LightGBM] [Info] Start training from score 0.111701
no_radar_reflect
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006903 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_w

In [10]:
# 3-3(추가). 강우변수군 전체 영향력 >> 개별강우변수군 영향력의 합  -> 강우변수군 변수간 상호작용 확인: 상관계수 필요
print("피어슨_상관계수: 강우변수군")
print(
    train[rain_cols + ['target_maxrise_6h']]
    .corr(method='pearson')
    .round(2)
    .to_csv(sep='\t')
)
print()

print("스피어만_상관계수: 강우변수군")
print(
    train[rain_cols + ['target_maxrise_6h']]
    .corr(method='spearman')
    .round(2)
    .to_csv(sep='\t')
)

print()

피어슨_상관계수: 강우변수군
	imerg_rain_1h	imerg_rain_6h	imerg_rain_24h	sfc_rain_1h	sfc_rain_6h	sfc_rain_24h	radar_reflectivity_mean	radar_reflectivity_p95	radar_reflectivity_ge20_fraction	target_maxrise_6h
imerg_rain_1h	1.0	0.65	0.4	0.36	0.29	0.2	0.49	0.42	0.47	0.19
imerg_rain_6h	0.65	1.0	0.69	0.42	0.58	0.42	0.5	0.43	0.49	0.2
imerg_rain_24h	0.4	0.69	1.0	0.29	0.5	0.72	0.34	0.31	0.33	0.16
sfc_rain_1h	0.36	0.42	0.29	1.0	0.62	0.39	0.45	0.41	0.43	0.26
sfc_rain_6h	0.29	0.58	0.5	0.62	1.0	0.7	0.38	0.35	0.36	0.25
sfc_rain_24h	0.2	0.42	0.72	0.39	0.7	1.0	0.25	0.26	0.24	0.19
radar_reflectivity_mean	0.49	0.5	0.34	0.45	0.38	0.25	1.0	0.91	0.92	0.19
radar_reflectivity_p95	0.42	0.43	0.31	0.41	0.35	0.26	0.91	1.0	0.82	0.18
radar_reflectivity_ge20_fraction	0.47	0.49	0.33	0.43	0.36	0.24	0.92	0.82	1.0	0.18
target_maxrise_6h	0.19	0.2	0.16	0.26	0.25	0.19	0.19	0.18	0.18	1.0


스피어만_상관계수: 강우변수군
	imerg_rain_1h	imerg_rain_6h	imerg_rain_24h	sfc_rain_1h	sfc_rain_6h	sfc_rain_24h	radar_reflectivity_mean	radar_reflectivity_p95	ra

In [13]:
# rmse 결과 df로 보기

result_df = pd.DataFrame(
result.items(),
columns=['model', 'rmse']
)

result_df['from_base_model'] = round((result_df['rmse'] - 0.2133), 4)

result_df

,model,rmse,from_base_model
0,no_wl_cols,0.3059,0.0926
1,no_rain_cols,0.2267,0.0134
2,no_env_cols,0.2161,0.0028
3,no_wl_t,0.2142,0.0009
4,no_wl_t_minus_1h,0.2161,0.0028
5,no_wl_t_minus_3h,0.2153,0.0020
6,no_wl_t_minus_6h,0.2244,0.0111
7,no_wl_t_minus_12h,0.2133,0.0000
8,no_wl_t_minus_24h,0.2136,0.0003
9,no_imerg,0.2144,0.0011
